# Exercicio

### Histórico de Navegação do Navegador
* Você foi contratado por uma startup que está desenvolvendo um navegador web do zero. A primeira funcionalidade a ser entregue é a navegação por histórico: o usuário precisa conseguir voltar para páginas" visitadas anteriormente e avançar novamente, exatamente como nos navegadores que usamos no dia a dia.
* O time de produto definiu o comportamento esperado:
* Quando o usuário visita uma página nova, ela é adicionada ao histórico e passa a ser a página atual.
* O botão Voltar leva o usuário à página anterior. A página atual não é perdida - ela fica guardada para permitir o Avançar.
* O botão Avançar refaz uma navegação que foi desfeita pelo Voltar.
* Regra crítica: quando o usuário visita uma página nova, todo o histórico de "Avançar" é descartado. Isso acontece porque, ao navegar para um lugar novo, os navegadores reais invalidam as páginas que estavam "à frente"

#### O problema
* Implemente, em linguagem C, um sistema de navegação com histórico usando pilha dinâmica (implementada com lista encadeada e alocação de memória via malloc/free). O sistema deve suportar as seguintes operações
* Operação visitar (url)
**comportamento Define url como página atual e a adiciona ao histórico. Esvazia o histórico de avançar.**
* operacao voltar()
**comportamento Retorna à página anterior. A página atual vai para o histórico de avançar. Se não houver página anterior, nada acontece**(exibe aviso).
* operacao avancar()
**comportamento Refaz a navegação desfeita. A página volta a ser a atual. Se não houver página para avançar, nada acontece (exibe aviso).**
* exibirAtual()
**Mostra a página atual na tela.**

In [ ]:
#include <stdbool.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct No {
    char *url;
    struct No *prox;
} No;

typedef struct {
    No *topo;
} PilhaURLs;

typedef struct {
    char *atual;
    PilhaURLs voltar;
    PilhaURLs avancar;
} Navegador;

char *duplicarString(const char *texto) {
    char *copia = malloc(strlen(texto) + 1);

    if (copia != NULL) {
        strcpy(copia, texto);
    }

    return copia;
}

void inicializarPilha(PilhaURLs *pilha) {
    pilha->topo = NULL;
}

bool pilhaVazia(const PilhaURLs *pilha) {
    return pilha->topo == NULL;
}

bool empilhar(PilhaURLs *pilha, char *url) {
    No *novo = malloc(sizeof(No));

    if (novo == NULL) {
        return false;
    }

    novo->url = url;
    novo->prox = pilha->topo;
    pilha->topo = novo;

    return true;
}

char *desempilhar(PilhaURLs *pilha) {
    if (pilhaVazia(pilha)) {
        return NULL;
    }

    No *removido = pilha->topo;
    char *url = removido->url;

    pilha->topo = removido->prox;
    free(removido);

    return url;
}

void limparPilha(PilhaURLs *pilha) {
    char *url;

    while ((url = desempilhar(pilha)) != NULL) {
        free(url);
    }
}

void inicializarNavegador(Navegador *navegador) {
    navegador->atual = NULL;
    inicializarPilha(&navegador->voltar);
    inicializarPilha(&navegador->avancar);
}

void visitar(Navegador *navegador, const char *url) {
    char *novaUrl = duplicarString(url);

    if (novaUrl == NULL) {
        printf("Erro ao alocar memoria.\n");
        return;
    }

    if (navegador->atual != NULL &&
        !empilhar(&navegador->voltar, navegador->atual)) {
        free(novaUrl);
        printf("Erro ao alocar memoria.\n");
        return;
    }

    navegador->atual = novaUrl;
    limparPilha(&navegador->avancar);

    printf("Visitando: %s\n", navegador->atual);
}

void voltar(Navegador *navegador) {
    if (pilhaVazia(&navegador->voltar)) {
        printf("Nao ha pagina anterior.\n");
        return;
    }

    if (!empilhar(&navegador->avancar, navegador->atual)) {
        printf("Erro ao alocar memoria.\n");
        return;
    }

    navegador->atual = desempilhar(&navegador->voltar);
    printf("Pagina atual: %s\n", navegador->atual);
}

void avancar(Navegador *navegador) {
    if (pilhaVazia(&navegador->avancar)) {
        printf("Nao ha pagina para avancar.\n");
        return;
    }

    if (!empilhar(&navegador->voltar, navegador->atual)) {
        printf("Erro ao alocar memoria.\n");
        return;
    }

    navegador->atual = desempilhar(&navegador->avancar);
    printf("Pagina atual: %s\n", navegador->atual);
}

void exibirAtual(const Navegador *navegador) {
    if (navegador->atual == NULL) {
        printf("Nenhuma pagina aberta.\n");
    } else {
        printf("Pagina atual: %s\n", navegador->atual);
    }
}

void liberarNavegador(Navegador *navegador) {
    free(navegador->atual);
    limparPilha(&navegador->voltar);
    limparPilha(&navegador->avancar);
}

int main(void) {
    Navegador navegador;
    int opcao;
    char url[256];

    inicializarNavegador(&navegador);

    do {
        printf("\n1 - Visitar\n");
        printf("2 - Voltar\n");
        printf("3 - Avancar\n");
        printf("4 - Exibir pagina atual\n");
        printf("0 - Sair\n");
        printf("Opcao: ");
        scanf("%d", &opcao);

        switch (opcao) {
            case 1:
                printf("URL: ");
                scanf("%255s", url);
                visitar(&navegador, url);
                break;
            case 2:
                voltar(&navegador);
                break;
            case 3:
                avancar(&navegador);
                break;
            case 4:
                exibirAtual(&navegador);
                break;
            case 0:
                break;
            default:
                printf("Opcao invalida.\n");
        }
    } while (opcao != 0);

    liberarNavegador(&navegador);
    return 0;
}


1 - Visitar
2 - Voltar
3 - Avancar
4 - Exibir pagina atual
0 - Sair
Opcao: 

# Explicação do código: histórico de navegação em C

## 1) Visão geral

Este programa simula um navegador web simples com três partes principais:

- `atual`: página que está aberta agora
- `voltar`: histórico das páginas anteriores
- `avancar`: páginas que foram desfeitas pelo botão "Voltar"

A estrutura usada para guardar o histórico é uma pilha dinâmica, ou seja, funciona como uma stack: o último item inserido é o primeiro a sair.

---

## 2) Estruturas básicas

### `No`
```c
typedef struct No {
    char *url;
    struct No *prox;
} No;
```

Cada `No` representa um item da pilha.

- `url`: guarda a string da página, como "google.com"
- `prox`: aponta para o próximo nó da lista

Isso cria uma lista encadeada, que é a base da pilha.

### `PilhaURLs`
```c
typedef struct {
    No *topo;
} PilhaURLs;
```

`topo` aponta para o elemento mais recente da pilha.

### `Navegador`
```c
typedef struct {
    char *atual;
    PilhaURLs voltar;
    PilhaURLs avancar;
} Navegador;
```

O navegador guarda:

- página atual
- histórico de páginas para voltar
- histórico de páginas para avançar

---

## 3) Função `duplicarString`

```c
char *duplicarString(const char *texto) {
    char *copia = malloc(strlen(texto) + 1);

    if (copia != NULL) {
        strcpy(copia, texto);
    }

    return copia;
}
```

Essa função cria uma cópia da string da URL.

### Por que isso é necessário?
Porque a URL precisa ser armazenada em memória própria. Se você apenas guarda um ponteiro para uma variável local, pode haver erro de memória ou perda de dados.

### Exemplo
```c
const char *site = "youtube.com";
char *copia = duplicarString(site);
```

Agora `copia` contém uma cópia separada de `site`.

---

## 4) Função `inicializarPilha`

```c
void inicializarPilha(PilhaURLs *pilha) {
    pilha->topo = NULL;
}
```

Essa função deixa a pilha vazia, apontando `topo` para `NULL`.

---

## 5) Função `pilhaVazia`

```c
bool pilhaVazia(const PilhaURLs *pilha) {
    return pilha->topo == NULL;
}
```

Retorna `true` se a pilha estiver vazia.

---

## 6) Função `empilhar`

```c
bool empilhar(PilhaURLs *pilha, char *url) {
    No *novo = malloc(sizeof(No));

    if (novo == NULL) {
        return false;
    }

    novo->url = url;
    novo->prox = pilha->topo;
    pilha->topo = novo;

    return true;
}
```

Essa função coloca uma nova URL no topo da pilha.

### Como funciona?
1. cria um novo nó
2. coloca a URL no nó
3. faz o novo nó apontar para o topo antigo
4. atualiza o topo para o novo nó

### Exemplo visual
Se a pilha estiver assim:

```text
topo -> A -> B -> C -> NULL
```

Ao empilhar `D`, ela vira:

```text
topo -> D -> A -> B -> C -> NULL
```

Ou seja, o último item empilhado fica em cima.

---

## 7) Função `desempilhar`

```c
char *desempilhar(PilhaURLs *pilha) {
    if (pilhaVazia(pilha)) {
        return NULL;
    }

    No *removido = pilha->topo;
    char *url = removido->url;

    pilha->topo = removido->prox;
    free(removido);

    return url;
}
```

Essa função remove o elemento do topo e retorna a URL.

### Exemplo
Se a pilha for:

```text
topo -> D -> A -> B -> C -> NULL
```

Ao desempilhar, o valor retornado é `D` e a pilha fica:

```text
topo -> A -> B -> C -> NULL
```

---

## 8) Função `limparPilha`

```c
void limparPilha(PilhaURLs *pilha) {
    char *url;

    while ((url = desempilhar(pilha)) != NULL) {
        free(url);
    }
}
```

Essa função remove todos os elementos da pilha e libera a memória das strings.

### Importante
`desempilhar` libera só o nó, mas não a string. Então aqui a string também é liberada com `free(url)`.

---

## 9) Função `inicializarNavegador`

```c
void inicializarNavegador(Navegador *navegador) {
    navegador->atual = NULL;
    inicializarPilha(&navegador->voltar);
    inicializarPilha(&navegador->avancar);
}
```

Cria o navegador em estado inicial:

- sem página atual
- sem histórico de voltar
- sem histórico de avançar

---

## 10) Função `visitar`

```c
void visitar(Navegador *navegador, const char *url) {
    char *novaUrl = duplicarString(url);

    if (novaUrl == NULL) {
        printf("Erro ao alocar memoria.\n");
        return;
    }

    if (navegador->atual != NULL &&
        !empilhar(&navegador->voltar, navegador->atual)) {
        free(novaUrl);
        printf("Erro ao alocar memoria.\n");
        return;
    }

    navegador->atual = novaUrl;
    limparPilha(&navegador->avancar);

    printf("Visitando: %s\n", navegador->atual);
}
```

Essa função é o coração do navegador.

### Lógica
1. cria uma cópia da URL nova
2. se já existir uma página atual, a página atual é empilhada em `voltar`
3. define a nova página como `atual`
4. limpa a pilha `avancar`

### Por que limpar `avancar`?
Porque quando você abre uma nova página, o caminho para frente fica inválido, como em um navegador real.

### Exemplo
Se o histórico atual for:

```text
atual = google.com
voltar = []
avancar = [youtube.com]
```

Ao visitar `github.com`:

- `google.com` vai para `voltar`
- `atual` vira `github.com`
- `avancar` é esvaziada

Resultado:

```text
atual = github.com
voltar = [google.com]
avancar = []
```

---

## 11) Função `voltar`

```c
void voltar(Navegador *navegador) {
    if (pilhaVazia(&navegador->voltar)) {
        printf("Nao ha pagina anterior.\n");
        return;
    }

    if (!empilhar(&navegador->avancar, navegador->atual)) {
        printf("Erro ao alocar memoria.\n");
        return;
    }

    navegador->atual = desempilhar(&navegador->voltar);
    printf("Pagina atual: %s\n", navegador->atual);
}
```

### Como funciona?
- se não houver página anterior, mostra aviso
- caso contrário:
  - a página atual é salva em `avancar`
  - a última página da pilha `voltar` vai para `atual`

### Exemplo
Suponha:

```text
atual = github.com
voltar = [google.com, youtube.com]
avancar = []
```

Ao chamar `voltar()`:

- empilha `github.com` em `avancar`
- retira `youtube.com` de `voltar`
- `youtube.com` vira a página atual

Resultado:

```text
atual = youtube.com
voltar = [google.com]
avancar = [github.com]
```

---

## 12) Função `avancar`

```c
void avancar(Navegador *navegador) {
    if (pilhaVazia(&navegador->avancar)) {
        printf("Nao ha pagina para avancar.\n");
        return;
    }

    if (!empilhar(&navegador->voltar, navegador->atual)) {
        printf("Erro ao alocar memoria.\n");
        return;
    }

    navegador->atual = desempilhar(&navegador->avancar);
    printf("Pagina atual: %s\n", navegador->atual);
}
```

Essa função faz o contrário de `voltar()`.

### Como funciona?
- se não houver nada em `avancar`, mostra aviso
- se houver:
  - a página atual vai para `voltar`
  - a página mais recente de `avancar` volta para `atual`

### Exemplo
Se o estado for:

```text
atual = youtube.com
voltar = [google.com]
avancar = [github.com]
```

Ao chamar `avancar()`:

- empilha `youtube.com` em `voltar`
- tira `github.com` de `avancar`
- `github.com` vira a página atual

Resultado:

```text
atual = github.com
voltar = [google.com, youtube.com]
avancar = []
```

---

## 13) Função `exibirAtual`

```c
void exibirAtual(const Navegador *navegador) {
    if (navegador->atual == NULL) {
        printf("Nenhuma pagina aberta.\n");
    } else {
        printf("Pagina atual: %s\n", navegador->atual);
    }
}
```

Essa função apenas mostra a página atual na tela.

### Se não houver página:
```text
Nenhuma pagina aberta.
```

---

## 14) Função `liberarNavegador`

```c
void liberarNavegador(Navegador *navegador) {
    free(navegador->atual);
    limparPilha(&navegador->voltar);
    limparPilha(&navegador->avancar);
}
```

Essa função libera toda a memória usada pelo navegador:

- página atual
- histórico de voltar
- histórico de avançar

Isso impede vazamento de memória.

---

## 15) Função `main`

```c
int main(void) {
    Navegador navegador;
    int opcao;
    char url[256];

    inicializarNavegador(&navegador);

    do {
        printf("\n1 - Visitar\n");
        printf("2 - Voltar\n");
        printf("3 - Avancar\n");
        printf("4 - Exibir pagina atual\n");
        printf("0 - Sair\n");
        printf("Opcao: ");
        scanf("%d", &opcao);

        switch (opcao) {
            case 1:
                printf("URL: ");
                scanf("%255s", url);
                visitar(&navegador, url);
                break;
            case 2:
                voltar(&navegador);
                break;
            case 3:
                avancar(&navegador);
                break;
            case 4:
                exibirAtual(&navegador);
                break;
            case 0:
                break;
            default:
                printf("Opcao invalida.\n");
        }
    } while (opcao != 0);

    liberarNavegador(&navegador);
    return 0;
}
```

Essa parte cria o menu do programa.

### Opções:
- `1`: visitar página
- `2`: voltar
- `3`: avançar
- `4`: exibir página atual
- `0`: sair

---

## 16) Exemplo de execução

### Fluxo
```text
visitar("google.com")
visitar("youtube.com")
visitar("github.com")
voltar()
voltar()
avancar()
```

### Resultado

#### 1) visitar("google.com")
```text
atual = google.com
voltar = []
avancar = []
```

#### 2) visitar("youtube.com")
```text
atual = youtube.com
voltar = [google.com]
avancar = []
```

#### 3) visitar("github.com")
```text
atual = github.com
voltar = [youtube.com, google.com]
avancar = []
```

#### 4) voltar()
```text
atual = youtube.com
voltar = [google.com]
avancar = [github.com]
```

#### 5) voltar()
```text
atual = google.com
voltar = []
avancar = [youtube.com, github.com]
```

#### 6) avancar()
```text
atual = youtube.com
voltar = [google.com]
avancar = [github.com]
```

---

## 17) Ideia lógica final

O código funciona como um navegador real:

- visitar uma nova página = nova página atual
- voltar = página anterior
- avançar = página que estava escondida
- visitar uma nova página = limpa o futuro (`avancar`)

Ou seja, ele reproduz o comportamento de históricos web usando pilhas implementadas com listas encadeadas.

---

## 18) Conclusão

Esse programa ensina vários conceitos importantes em C:

- structs
- ponteiros
- memória dinâmica com `malloc`
- listas encadeadas
- pilhas
- liberação de memória
- manipulação de strings

É um ótimo exemplo de como implementar estruturas de dados que simulam comportamento real de sistemas do mundo cotidiano.



1 - Visitar
2 - Voltar
3 - Avancar
4 - Exibir pagina atual
0 - Sair
Opcao: 